# Tutorial: Model Evaluation

> **Type:** Tutorial — follow along step by step. Your only job is to make it work.  
> **Time:** ~2 hours  
> **Prerequisites:** Modules 11–13 (supervised learning, ensembles, cross-validation)  
> **What you'll build:** A complete evaluation report for both a classifier and a regressor — confusion matrix, ROC curve, PR curve, calibration curve, and residual analysis — plus a framework for choosing the right metric for any business problem.

---

## What We're Building

We'll train two models:
1. A **classifier** on the breast cancer dataset — then evaluate it every way possible
2. A **regressor** on synthetic housing data — then diagnose where it fails

By the end you'll be able to produce a professional evaluation report for any model.

## Step 0: Setup

In [ ]:
# !pip install scikit-learn matplotlib seaborn numpy pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, make_regression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    log_loss,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
SEED = 42
np.random.seed(SEED)
print('Setup complete.')

---
## Part 1: Classification Evaluation

### Step 1: Load Data and Train a Classifier

In [ ]:
# Breast cancer dataset — 569 samples, 30 features, binary target (malignant / benign)
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
target_names = data.target_names  # ['malignant', 'benign']

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Target classes: {target_names}")
print(f"Class balance: {pd.Series(y).value_counts().to_dict()}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Train a logistic regression (outputs calibrated probabilities by default)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=SEED)
model.fit(X_train_s, y_train)

y_pred       = model.predict(X_test_s)
y_proba      = model.predict_proba(X_test_s)[:, 1]  # probability of positive class

print(f"Model trained. Test set size: {len(y_test)}")

### Step 2: The Confusion Matrix

Always start here. Every other metric is derived from this table.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=target_names, yticklabels=target_names,
    ax=ax
)
ax.set_xlabel('Predicted label', fontsize=12)
ax.set_ylabel('True label', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")

> **What just happened?** The heatmap shows the four cells of the confusion matrix. Dark blue = many predictions in that cell. FN (bottom-left) is the dangerous one for medical diagnosis — those are the cancers we missed.

### Step 3: Compute All Classification Metrics

In [ ]:
metrics = {
    'Accuracy':       accuracy_score(y_test, y_pred),
    'Precision':      precision_score(y_test, y_pred),
    'Recall':         recall_score(y_test, y_pred),
    'F1':             f1_score(y_test, y_pred),
    'ROC-AUC':        roc_auc_score(y_test, y_proba),
    'Log Loss':       log_loss(y_test, y_proba),
    'Avg Precision':  average_precision_score(y_test, y_proba),
}

print("=== Classification Metrics ===")
for name, val in metrics.items():
    print(f"  {name:20s}: {val:.4f}")

print("\n=== Full Classification Report ===")
print(classification_report(y_test, y_pred, target_names=target_names))

### Step 4: ROC Curve and PR Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- ROC Curve ---
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

axes[0].plot(fpr, tpr, color='#4E79A7', linewidth=2,
             label=f'Logistic Regression (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier (AUC = 0.5)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend()

# --- Precision-Recall Curve ---
precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_test, y_proba)
avg_prec = average_precision_score(y_test, y_proba)

axes[1].plot(recall_vals, precision_vals, color='#E15759', linewidth=2,
             label=f'Logistic Regression (AP = {avg_prec:.3f})')
baseline = y_test.mean()
axes[1].axhline(baseline, color='grey', linestyle='--', linewidth=1,
                label=f'No-skill baseline ({baseline:.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

> **What just happened?** The ROC curve shows the trade-off between catching true positives and generating false alarms at every threshold. The PR curve shows the precision-recall trade-off — more useful when one class is rare.

### Step 5: Moving the Decision Threshold

In [ ]:
# The default threshold is 0.5 — but we can move it
thresholds = np.arange(0.1, 0.95, 0.05)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, precisions, label='Precision', color='#4E79A7', linewidth=2)
ax.plot(thresholds, recalls,    label='Recall',    color='#E15759', linewidth=2)
ax.plot(thresholds, f1s,        label='F1',        color='#F28E2B', linewidth=2, linestyle='--')
ax.axvline(0.5, color='grey', linestyle=':', linewidth=1.5, label='Default threshold (0.5)')

best_f1_idx = np.argmax(f1s)
best_t = thresholds[best_f1_idx]
ax.axvline(best_t, color='green', linestyle=':', linewidth=1.5, 
           label=f'Best F1 threshold ({best_t:.2f})')

ax.set_xlabel('Decision threshold')
ax.set_ylabel('Score')
ax.set_title('Precision, Recall, and F1 vs Decision Threshold', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Best F1 at threshold {best_t:.2f}: F1={f1s[best_f1_idx]:.3f}, "
      f"Precision={precisions[best_f1_idx]:.3f}, Recall={recalls[best_f1_idx]:.3f}")

> **What just happened?** As we increase the threshold, precision rises and recall falls. The best F1 score isn't always at 0.5. For cancer screening you'd shift the threshold left (lower) to maximise recall — catching every cancer matters more than avoiding false alarms.

### Step 6: Calibration Curve

In [ ]:
# Train a second model (Random Forest) to compare calibration
rf = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf.fit(X_train_s, y_train)
rf_proba = rf.predict_proba(X_test_s)[:, 1]

fig, ax = plt.subplots(figsize=(7, 6))

for proba, name, colour in [
    (y_proba,  'Logistic Regression', '#4E79A7'),
    (rf_proba, 'Random Forest',       '#E15759'),
]:
    fraction_of_positives, mean_predicted = calibration_curve(
        y_test, proba, n_bins=10
    )
    ax.plot(mean_predicted, fraction_of_positives, 
            marker='o', linewidth=2, color=colour, label=name)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Perfectly calibrated')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Curve (Reliability Diagram)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()

> **What just happened?** A perfectly calibrated model's curve lies on the diagonal. Logistic regression is usually well-calibrated. Random Forests tend to be overconfident at the extremes (sigmoid-shaped curve). When calibration matters — risk scores, pricing — you can fix it with `CalibratedClassifierCV`.

---
## Part 2: Regression Evaluation

### Step 7: Generate Synthetic Housing Data and Train a Regressor

In [ ]:
# Synthetic regression dataset with injected outliers
X_reg, y_reg = make_regression(
    n_samples=500, n_features=10, noise=25, random_state=SEED
)

# Inject 10 severe outliers
outlier_idx = np.random.choice(len(y_reg), 10, replace=False)
y_reg[outlier_idx] += np.random.choice([-500, 500], 10)

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=SEED)

reg = RandomForestRegressor(n_estimators=100, random_state=SEED)
reg.fit(X_tr, y_tr)
y_te_pred = reg.predict(X_te)

print(f"Regression dataset: {X_reg.shape[0]} samples, {X_reg.shape[1]} features")
print(f"10 outliers injected at indices: {sorted(outlier_idx[:5])}...")

### Step 8: Compute Regression Metrics

In [ ]:
mse  = mean_squared_error(y_te, y_te_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_te, y_te_pred)
r2   = r2_score(y_te, y_te_pred)

# MAPE — be careful when y contains zeros
mask = y_te != 0
mape = np.mean(np.abs((y_te[mask] - y_te_pred[mask]) / y_te[mask])) * 100

print("=== Regression Metrics ===")
print(f"  MSE  : {mse:>10.2f}")
print(f"  RMSE : {rmse:>10.2f}  (same units as target)")
print(f"  MAE  : {mae:>10.2f}  (less sensitive to outliers)")
print(f"  MAPE : {mape:>10.2f}%")
print(f"  R²   : {r2:>10.4f}")

print(f"\nRMSE / MAE ratio: {rmse/mae:.2f}x")
print("(Ratio > 1 means outliers are inflating RMSE — they're present here)")

### Step 9: Predicted vs Actual and Residual Analysis

In [ ]:
residuals = y_te - y_te_pred
outlier_mask = np.abs(residuals) > 200  # flag extreme residuals

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Predicted vs Actual ---
axes[0].scatter(y_te[~outlier_mask], y_te_pred[~outlier_mask],
                alpha=0.5, color='#4E79A7', s=30, label='Normal')
axes[0].scatter(y_te[outlier_mask],  y_te_pred[outlier_mask],
                alpha=0.9, color='#E15759', s=80, label='Outlier', zorder=5)
lims = [min(y_te.min(), y_te_pred.min()), max(y_te.max(), y_te_pred.max())]
axes[0].plot(lims, lims, 'k--', linewidth=1, label='Perfect predictions')
axes[0].set_xlabel('Actual values')
axes[0].set_ylabel('Predicted values')
axes[0].set_title('Predicted vs Actual', fontweight='bold')
axes[0].legend()

# --- Residuals vs Predicted ---
axes[1].scatter(y_te_pred[~outlier_mask], residuals[~outlier_mask],
                alpha=0.5, color='#4E79A7', s=30)
axes[1].scatter(y_te_pred[outlier_mask],  residuals[outlier_mask],
                alpha=0.9, color='#E15759', s=80, label='Outlier', zorder=5)
axes[1].axhline(0, color='black', linewidth=1.5)
axes[1].set_xlabel('Predicted values')
axes[1].set_ylabel('Residuals (actual − predicted)')
axes[1].set_title('Residuals vs Predicted', fontweight='bold')

# --- Residual Distribution ---
axes[2].hist(residuals[~outlier_mask], bins=30, color='#4E79A7', alpha=0.7, label='Normal')
axes[2].hist(residuals[outlier_mask],  bins=5,  color='#E15759', alpha=0.9, label='Outlier')
axes[2].axvline(0, color='black', linewidth=1.5)
axes[2].set_xlabel('Residual')
axes[2].set_ylabel('Count')
axes[2].set_title('Residual Distribution', fontweight='bold')
axes[2].legend()

plt.suptitle(f'Regression Evaluation  |  RMSE={rmse:.1f}  MAE={mae:.1f}  R²={r2:.3f}',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('regression_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

> **What just happened?** The red dots are the injected outliers — they appear as extreme residuals in the middle plot and as a long tail in the histogram. RMSE is inflated by these; MAE is less affected. That's why the RMSE/MAE ratio is a quick outlier diagnostic.

### Step 10: The Business Context Question

You're building a fraud detection model. The dataset has 1000 transactions, 20 are fraudulent.
A fraud analyst can review 50 flagged transactions per day.

In [ ]:
# Simulate the business trade-off
total_txns   = 1000
n_fraud      = 20
analyst_cap  = 50   # can review this many flagged txns per day
fraud_loss   = 500  # dollars lost per missed fraud
review_cost  = 10   # cost per reviewed transaction

# Scenario A: Optimise for Accuracy (flag everything above 0.5)
#   Model flags 100 transactions, catches 18/20 frauds
flagged_A, caught_A = 100, 18

# Scenario B: Optimise for Precision (flag only high-confidence)
#   Model flags 25 transactions, catches 15/20 frauds
flagged_B, caught_B = 25, 15

# Scenario C: Optimise for Recall (flag everything remotely suspicious)
#   Model flags 200 transactions, catches 20/20 frauds
flagged_C, caught_C = 200, 20

print("=== Business Impact of Metric Choice ===")
print(f"{'Scenario':<12} {'Flagged':>10} {'Caught':>8} {'Reviewed':>10} {'Missed $':>10} {'Review $':>10} {'Net Loss':>10}")
print("-" * 72)

for name, flagged, caught in [
    ('Accuracy',  flagged_A, caught_A),
    ('Precision', flagged_B, caught_B),
    ('Recall',    flagged_C, caught_C),
]:
    reviewed     = min(flagged, analyst_cap)
    missed_fraud = n_fraud - caught
    missed_cost  = missed_fraud * fraud_loss
    rev_cost     = reviewed * review_cost
    net_loss     = missed_cost + rev_cost
    print(f"{name:<12} {flagged:>10} {caught:>8} {reviewed:>10} {missed_cost:>10} {rev_cost:>10} {net_loss:>10}")

print("\n=> The 'right' metric depends on fraud_loss, analyst_cap, and review_cost.")
print("   There is no universal best metric — only context-correct metrics.")

---
## What You Built

A complete evaluation toolkit: confusion matrix, all classification metrics, ROC and PR curves, threshold analysis, calibration curves, and a full regression diagnostic suite. You also worked through the business impact calculation that bridges ML metrics to real-world decisions.

## What's Next

- Read `explanation.md` for the formulas and the business metric gap concept.
- Use `howto.md` when you need a quick metric recipe on your own data.
- Keep `reference.md` open — the metric selection guide is one you'll consult repeatedly.
- **Extension challenge:** Re-run the classification section with an imbalanced dataset. Use `sklearn.datasets.make_classification` with `weights=[0.95, 0.05]` and observe how accuracy flatlines at 95% even for a useless model.

---

**Next module:** `17_hyperparameter_optimization` — because once you can evaluate a model, the next question is how to make it better.